# **01_best_model.ipynb**

**Programmers:**
* Albonia, Jade Lorenz M.
* Caspe, Mark Vincent G.
* Rivera, Rei Djemf M.
* Velante, Kamilah Kaye M.
* Villegas, Jedidiah S.

**Date Written:** August 2025

**Date Revised:** December 2025

---

### **System Context**
This notebook represents the critical transition from **Research (Phase 1 & 2)** to **Deployment (Phase 3)**. While previous notebooks focused on statistical validation (K-Fold Cross-Validation) to prove the hypothesis, this pipeline is dedicated to generating the single, definitive **Model Artifact (`.pth`)** that will be embedded into the Android mobile application.

### **Purpose**
To train the optimal architecture (**A-EYE 4-Ring**) on the complete training dataset using the optimal hyperparameters identified during the ablation study (Epochs: 30, Learning Rate: 2e-4). This ensures the deployed model has learned from the maximum available data variance without overfitting.

---

### **Technical Architecture (Data Structures & Algorithms)**

**1. Data Structures**
* **Production Weights (`state_dict`):** The serialized dictionary of ~2.3 million parameters that constitutes the trained "brain" of the A-EYE system.
* **Final Assets Archive (`.zip`):** A consolidated binary package containing the model weights, configuration logs, and performance graphs, structured for direct handoff to the mobile engineering team.

**2. Algorithms**
* **Full-Scale Optimization:** Unlike the K-Fold approach which splits data to test stability, this routine utilizes the **Full Training Set (100%)** to optimize the model's weights.
* **Production Evaluation:** Performs a final "Sanity Check" on the held-out Test Set to ensure the production candidate meets the minimum clinical accuracy thresholds (Safety Gate).

**3. Control Flow**
* **Sequential Pipeline:**
    1.  **Environment Provisioning:** Clones repo and installs dependencies.
    2.  **Training (`final_train.py`):** Executes the training loop with `epochs=30` and `lr=2e-4`.
    3.  **Evaluation (`final_evaluate.py`):** Generates the final confusion matrix.
    4.  **Packaging:** Compresses the `saved_models` directory for export.

---

## Step 1: Primary Setup

In [ ]:
# This cell prepares the Colab environment.
import os
import sys
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

REPO_URL = "https://github.com/its-levi0sa/a-eye-cataract-maturity-classification-tool.git"
PROJECT_DIR = "A-EYE"

# --- Clone or Pull Latest Code ---
if os.path.exists(PROJECT_DIR):
    print("Repository already exists. Pulling latest changes...")
    %cd {PROJECT_DIR}
    !git pull
else:
    print("Cloning repository...")
    !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}

# --- Configure Paths and Install Dependencies ---
if os.path.abspath('.') not in sys.path:
    sys.path.insert(0, os.path.abspath('.'))

!pip install -q -r requirements.txt

print("\n\n✅ Environment setup complete. All scripts and data are ready.")

## Step 2: Run Final Training & Evaluation

In [ ]:
import os

# --- Define Model and Data Paths ---
MODEL_TYPE = 'aeye'
NUM_RINGS = 4
EPOCHS = 30                       # Calculated from 5-fold average peak (30.4)
LEARNING_RATE = 1e-4              # Optimal LR
TRAIN_DATA_DIR = 'data/train'
TEST_DATA_DIR = 'data/test'
SAVE_DIR = 'saved_models/final'
FINAL_MODEL_NAME = f"{MODEL_TYPE}_{NUM_RINGS}_rings_final_model.pth"
FINAL_MODEL_PATH = os.path.join(SAVE_DIR, FINAL_MODEL_NAME)
LOG_FILE = 'results/final_run_log.txt'

# Ensure the results directory exists
os.makedirs('results', exist_ok=True)

# --- 1. RUN FINAL TRAINING ---
print("-" * 45)
print(f"STARTING FINAL TRAINING: A-EYE ({NUM_RINGS} Rings)")
print(f"Settings: Epochs={EPOCHS}, LR={LEARNING_RATE}")
print("-" * 45)

!python scripts/final_train.py \
  --model_type {MODEL_TYPE} \
  --num_rings {NUM_RINGS} \
  --data_dir {TRAIN_DATA_DIR} \
  --save_dir {SAVE_DIR} \
  --epochs {EPOCHS} \
  --learning_rate {LEARNING_RATE} \
  | tee {LOG_FILE}

!echo -e "\n\n" >> {LOG_FILE}

# --- 2. RUN FINAL EVALUATION ---
print("\n" + "-" * 45)
print(f"STARTING FINAL EVALUATION on the Test Set")
print("-" * 45)

!python scripts/final_evaluate.py \
  --model_path {FINAL_MODEL_PATH} \
  --model_type {MODEL_TYPE} \
  --num_rings {NUM_RINGS} \
  --data_dir {TEST_DATA_DIR} \
  | tee -a {LOG_FILE}

## Step 3: Calibration

In [ ]:
%%writefile scripts/calibrate_single_model.py
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import glob
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))
from src.aeye_model import AEyeModel
from src.data_utils import get_transforms, AlbumentationsDataset

# --- CONFIGURATION ---
MODEL_PATH = 'saved_models/final/aeye_4_rings_final_model.pth'
DATA_DIR = 'data/train'
NUM_RINGS = 4
BATCH_SIZE = 1

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # 1. Load the single final trained model
    model_config = {'num_rings': NUM_RINGS, 'dims': [32, 64, 128, 160], 'embed_dim': 256}
    model = AEyeModel(model_config)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.to(device).eval()
    print(f"Successfully loaded single model for calibration from {MODEL_PATH}")

    # 2. Load the full training dataset
    image_paths = glob.glob(os.path.join(DATA_DIR, '*/*.[jp][pn]g'))
    dataset = AlbumentationsDataset(image_paths, labels=[0]*len(image_paths), transform=get_transforms(is_train=False))
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    print(f"Loaded {len(dataset)} images for calibration.")

    # 3. Loop through all data and collect statistics
    all_brightness_values, all_texture_values = [], []
    with torch.no_grad():
        for images, _ in tqdm(loader, desc="Calibrating Single Model"):
            images = images.to(device)
            _, tokens = model(images, return_tokens=True)

            tokens_np = tokens.squeeze(0).cpu().numpy()
            mean_rgb = (tokens_np[:, 0:3] * 0.5 + 0.5) * 255
            std_rgb = (tokens_np[:, 3:6] * 0.5) * 255
            all_brightness_values.append(np.mean(mean_rgb))
            all_texture_values.append(np.mean(std_rgb))

    # 4. Calculate percentiles
    p1_brightness = np.percentile(all_brightness_values, 1)
    p99_brightness = np.percentile(all_brightness_values, 99)
    p1_texture = np.percentile(all_texture_values, 1)
    p99_texture = np.percentile(all_texture_values, 99)

    # 5. Print results
    print("\n" + "="*50)
    print("--- CALIBRATION COMPLETE (SINGLE MODEL) ---")
    print("\nUse these constants for the single-model predict script:\n")
    print(f"P1_BRIGHTNESS = {p1_brightness:.2f}")
    print(f"P99_BRIGHTNESS = {p99_brightness:.2f}")
    print(f"P1_TEXTURE = {p1_texture:.2f}")
    print(f"P99_TEXTURE = {p99_texture:.2f}")
    print("\n" + "="*50)

if __name__ == '__main__':
    main()

In [ ]:
# This cell runs the single-model calibration
!python scripts/calibrate_single_model.py

## Step 4: Prediction

In [ ]:
%%writefile scripts/predict_single_model.py
import os
import argparse
import numpy as np
import torch
import cv2
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from src.aeye_model import AEyeModel
from src.data_utils import get_transforms

def generate_aeye_explanation(tokens_tensor, num_rings):
    """
    Generates a detailed, human-friendly report from the A-EYE model's
    internal statistics for a single image.
    """
    # --- CALIBRATED CONSTANTS ---
    P1_BRIGHTNESS = 50.82
    P99_BRIGHTNESS = 172.22
    P1_TEXTURE = 25.47
    P99_TEXTURE = 63.68

    # --- Data Extraction and Denormalization ---
    avg_tokens = tokens_tensor.squeeze(0).cpu().numpy()
    mean_rgb = (avg_tokens[:, 0:3] * 0.5 + 0.5) * 255
    std_rgb = (avg_tokens[:, 3:6] * 0.5) * 255

    # Calculate overall statistics for the entire lens area.
    overall_mean_brightness = np.mean(mean_rgb)
    overall_mean_texture = np.mean(std_rgb)

    # --- Proxy Calculation using Min-Max Normalization ---
    brightness_proxy = 100 * (overall_mean_brightness - P1_BRIGHTNESS) / (P99_BRIGHTNESS - P1_BRIGHTNESS)
    opacity_proxy = 100 * (overall_mean_texture - P1_TEXTURE) / (P99_TEXTURE - P1_TEXTURE)

    # Ensure the final percentages do not go outside the 0-100 range.
    brightness_proxy, opacity_proxy = np.clip([brightness_proxy, opacity_proxy], 0, 100)

    # Refine the density score to prevent false positives from light reflections.
    final_opacity_proxy = opacity_proxy
    if brightness_proxy < 5.0:
        final_opacity_proxy = opacity_proxy * (brightness_proxy / 5.0)

    # --- Construct the Final Report ---
    report = "\n" + "="*65 + "\n"
    report += "\t    A-EYE BEST MODEL EXPLAINABILITY REPORT\n"
    report += "="*65 + "\n"
    report += "Disclaimer: The following percentages are data-driven proxies derived\n"
    report += "from the model's statistics, not direct clinical measurements.\n\n"

    report += "--- Human-Readable Summary ---\n"
    report += f"   - Estimated Opacity Extent (Brightness): {brightness_proxy:.1f}%\n"
    report += f"   - Estimated Opacity Density (Texture):   {final_opacity_proxy:.1f}%\n\n"

    report += "--- Data-Driven Details for Thesis Discussion ---\n"
    for i in range(num_rings):
        # Calculate the average brightness and texture for each individual ring.
        mean_gray_per_ring = np.mean(mean_rgb[i])
        std_gray_per_ring = np.mean(std_rgb[i])
        report += f"   - Ring {i+1:02d}: Brightness={mean_gray_per_ring:6.2f}, Texture={std_gray_per_ring:6.2f}\n"

    report += "="*65 + "\n"
    return report

def predict(args):
    """Main function to load a single model and run prediction on an image."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Initialize the model with the correct architecture.
    model_config = {'dims': args.dims, 'embed_dim': args.embed_dim, 'num_rings': args.num_rings}
    model = AEyeModel(model_config)

    # Load the trained weights from the specified .pth file.
    model.load_state_dict(torch.load(args.model_path, map_location=device))
    model.to(device).eval()
    print(f"Loaded single model from '{args.model_path}'.")

    # Load and preprocess the input image.
    image = cv2.imread(args.image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    input_tensor = get_transforms(is_train=False)(image=image)['image'].unsqueeze(0).to(device)

    # Perform inference.
    with torch.no_grad():
        output, tokens = model(input_tensor, return_tokens=True)

    # Convert the model's raw output (logit) to a probability score.
    final_prob = torch.sigmoid(output).item()
    prediction = "Mature" if final_prob >= 0.5 else "Immature"

    # Display the final results.
    print("\n\t     --- PREDICTION RESULT (BEST MODEL) ---")
    print(f"Image:             {os.path.basename(args.image_path)}")
    print(f"Predicted Class:   {prediction}")
    print(f"Confidence Score:  {final_prob:.2%}")
    print(generate_aeye_explanation(tokens, args.num_rings))

if __name__ == '__main__':
    # --- Argument Parsing ---
    parser = argparse.ArgumentParser(description="Single Model Prediction Script")
    parser.add_argument('--model_path', required=True, help="Path to the single trained .pth model file.")
    parser.add_argument('--image_path', required=True, help="Path to the input image for prediction.")
    parser.add_argument('--num_rings', type=int, default=4, help="Number of rings used by the A-EYE model.")
    parser.add_argument('--dims', type=int, nargs='+', default=[32, 64, 128, 160])
    parser.add_argument('--embed_dim', type=int, default=256)
    args = parser.parse_args()
    predict(args)

In [ ]:
# --- This cell runs prediction on the single, final model ---
# Remember to paste the calibrated constants into the script above first!

!python scripts/predict_single_model.py \
    --model_path saved_models/final/aeye_4_rings_final_model.pth \
    --image_path data/test/mature/mature_056.png

!echo -e "\n\n"

!python scripts/predict_single_model.py \
    --model_path saved_models/final/aeye_4_rings_final_model.pth \
    --image_path data/test/immature/immature_056.png

## Step 5: Download Final Assets

In [ ]:
FINAL_ZIP = '/content/final_deployment_assets.zip'

print(f"Zipping final assets into {FINAL_ZIP}...")

# Zip the necessary assets/results
!zip -r {FINAL_ZIP} results/final_run_log.txt results/confusion_matrix_final_evaluation_results_aeye_4_rings_final_model.png saved_models/final/

print(f"✅ All final assets have been zipped into '{FINAL_ZIP}'.")
print("You can now download it from the file browser on the left.")